# Изменение скорости обучения в процессе обучения

Пример использования callback'ов [LearningRateScheduler](https://keras.io/api/callbacks/learning_rate_scheduler/) и [ReduceLROnPlateau](https://keras.io/api/callbacks/model_checkpoint/).

Чтобы запускать и редактировать код, сохраните копию этого ноутбука себе (Файл -> Создать копию на Диске). Свою копию вы сможете изменять и запускать.

Учебный курс "[Программирование глубоких нейронных сетей на Python](https://openedu.ru/course/urfu/PYDNN/)".

<a target="_blank" href="https://colab.research.google.com/github/sozykin/dlpython_course/blob/master/keras_callbacks/lr_scheduler.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



In [5]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Rescaling, Flatten
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Загружаем данные Fashion MNIST

In [2]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Создаем нейронную сеть

In [18]:
# Создаем последовательную модель
model = Sequential(
    [   
        # Слой, который преобразует матрицу 28х28 в плоский вектор
        Flatten(),
        # Слой нормализации
        Rescaling(1./255),
        # Входной полносвязный слой, 800 нейрона
        # Количество входов не указываем, Keras определяет автоматически
        # после первого запуска
        Dense(800, activation="relu"),
        # Выходной полносвязный слой, 10 нейронов (по количеству классов)
        Dense(10, activation="softmax")
    ]
)

Компилируем сеть

In [19]:
model.compile(loss="sparse_categorical_crossentropy", 
              optimizer="adam",
              metrics=["accuracy"])

## Создаем Callback для изменения параметра скорости обучения

Управляем значением параметра скорости обучения

Функция расчет параметра скорости обучения

In [7]:
def scheduler(epoch, lr):
    if epoch < 10:
        return 0.001    # Скорость обучения до 10 эпохи
    elif epoch < 20:
        return 0.0001   # Скорость обучения от 10 до 20 эпохи
    else:
        return 0.00001  # Скорость обучения после 20 эпохи

In [20]:
lr_scheduler = LearningRateScheduler(scheduler,
                                     verbose=1)

Уменьшаем скорость обучения если обучение остановилось

In [14]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', # Мониторим val_loss
                              patience=5,         # Ждем 5 эпох
                                                  # Если val_loss не уменьшается, 
                                                  # тогда умножаем скорость обучения на factor 
                              factor=0.5,         # На сколько умножать скорость обучения
                              min_lr=1e-7,        # Минимальное значение скорости обучения 
                              verbose=1),

## Запускаем обучение нейронной сети

In [21]:
history = model.fit(x_train,
                    y_train, 
                    batch_size=200, 
                    epochs=25, 
                    validation_split=0.2, 
                    verbose=2, 
                    callbacks=[lr_scheduler])


Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/25
240/240 - 6s - 27ms/step - accuracy: 0.8103 - loss: 0.5415 - val_accuracy: 0.8483 - val_loss: 0.4356 - learning_rate: 1.0000e-03

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/25
240/240 - 4s - 15ms/step - accuracy: 0.8617 - loss: 0.3891 - val_accuracy: 0.8662 - val_loss: 0.3791 - learning_rate: 1.0000e-03

Epoch 3: LearningRateScheduler setting learning rate to 0.001.
Epoch 3/25
240/240 - 4s - 15ms/step - accuracy: 0.8749 - loss: 0.3460 - val_accuracy: 0.8698 - val_loss: 0.3753 - learning_rate: 1.0000e-03

Epoch 4: LearningRateScheduler setting learning rate to 0.001.
Epoch 4/25
240/240 - 3s - 14ms/step - accuracy: 0.8831 - loss: 0.3167 - val_accuracy: 0.8819 - val_loss: 0.3320 - learning_rate: 1.0000e-03

Epoch 5: LearningRateScheduler setting learning rate to 0.001.
Epoch 5/25
240/240 - 4s - 15ms/step - accuracy: 0.8904 - loss: 0.2973 - val_accuracy: 0.8811 - val_loss: 0.3365 - learni

## Проверяем модель на тестовых данных

In [22]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8939 - loss: 0.3064


In [23]:
test_accuracy

0.8938999772071838